In [17]:
import xgboost as xgb
import sys
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow 
import mlflow.xgboost

In [18]:
# loading dfs'
train_df = pd.read_csv(r"Y:\code\regression-pipeline\data\processed\fe_train.csv")
val_df = pd.read_csv(r"Y:\code\regression-pipeline\data\processed\fe_val.csv")

target = "price"

X_train = train_df.drop(columns=[target])
Y_train = train_df[target]

X_val = val_df.drop(columns=[target])
Y_val = val_df[target]

In [21]:
# defining optuna objective function with mlflow

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(Y_val, Y_pred)))
        mae = float(mean_absolute_error(Y_val, Y_pred))
        r2 = float(r2_score(Y_val, Y_pred))

        # Log hyperparameters + metrics for tracking
        mlflow.log_params(params)
        mlflow.log_metrics({
            "rmse": rmse,
            "mae": mae,
            "r2": r2
            })

    return rmse

In [22]:
# run optuna study with mlflow

# force mlflow to use the root project mlruns folder 
mlflow.set_tracking_uri("file:///Y:/code/regression-pipeline/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print(f"Best params : {study.best_trial.params}")

[I 2026-03-15 21:59:32,982] A new study created in memory with name: no-name-67cd81be-0138-40d8-be26-6ba16f73ce4f
[I 2026-03-15 22:00:46,704] Trial 0 finished with value: 72058.02149661866 and parameters: {'n_estimators': 560, 'max_depth': 9, 'learning_rate': 0.029169699801986512, 'subsample': 0.6669218992293611, 'colsample_bytree': 0.9896840754440142, 'min_child_weight': 9, 'gamma': 4.608146730967567, 'reg_alpha': 7.12572941742712e-08, 'reg_lambda': 0.00045532835290608447}. Best is trial 0 with value: 72058.02149661866.
[I 2026-03-15 22:01:20,476] Trial 1 finished with value: 75063.92191217483 and parameters: {'n_estimators': 254, 'max_depth': 10, 'learning_rate': 0.08516498272209913, 'subsample': 0.7295149073041456, 'colsample_bytree': 0.7439608818850454, 'min_child_weight': 1, 'gamma': 4.148633477538378, 'reg_alpha': 0.07919073384039421, 'reg_lambda': 0.017612732435659386}. Best is trial 0 with value: 72058.02149661866.
[I 2026-03-15 22:01:38,768] Trial 2 finished with value: 71895.

Best params : {'n_estimators': 784, 'max_depth': 9, 'learning_rate': 0.04977270970455462, 'subsample': 0.6247988771264417, 'colsample_bytree': 0.6549630055412189, 'min_child_weight': 6, 'gamma': 3.6430537073058478, 'reg_alpha': 6.221029804460751e-07, 'reg_lambda': 0.0023533492154144513}


In [23]:
# training the final model with optimal params

best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, Y_train)

Y_pred = best_model.predict(X_val)

rmse = float(np.sqrt(mean_squared_error(Y_val, Y_pred)))
mae = float(mean_absolute_error(Y_val, Y_pred))
r2 = float(r2_score(Y_val, Y_pred))

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse" :  rmse, "mae" : mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 30657.64060071312
RMSE: 71248.10241492315
R²: 0.9607710543973559
